### Phase 5: Evaluation & Vergleich (Der Kern der Thesis)
Beweis des Mehrwerts zur Reduktion des **Sequence of Returns Risk**.
*   **Metriken:**
    *   **Maximum Drawdown (MDD):** Wie tief war der maximale Fall? (Wichtigstes Ziel).
    *   **Sharpe Ratio / Sortino Ratio:** Risiko-adjustierte Rendite.
    *   **Calmar Ratio:** Rendite im Verhältnis zum Drawdown.
    *   **Regime-Stabilität:** Wie oft schaltet das Modell um? (LSTMs neigen zum "Overfitting" und Rauschen).

In [1]:
import pandas as pd

# Daten aus dem data-Ordner laden
backtesting_results = pd.read_parquet("../data/04_backtesting_results_data.parquet")
backtesting_transaction_costs = pd.read_parquet("../data/04_backtesting_transaction_costs_data.parquet")

In [2]:
import pandas as pd
import numpy as np

# 1. Daten aus dem data-Ordner laden
test_df = pd.read_parquet("../data/03_test_df_data.parquet")
backtesting_results = pd.read_parquet("../data/04_backtesting_results_data.parquet")

# 2. Funktion zur Evaluation
def evaluate_strategies(results_df, trades_df, costs_df):
    stats = []
    
    for col in results_df.columns:
        equity_curve = results_df[col]
        daily_returns = equity_curve.pct_change().dropna()
        
        # 1. Total Return & CAGR (Annualisierte Rendite)
        total_return = (equity_curve.iloc[-1] / equity_curve.iloc[0]) - 1
        days = (equity_curve.index[-1] - equity_curve.index[0]).days
        cagr = (equity_curve.iloc[-1] / equity_curve.iloc[0])**(365.25/days) - 1
        
        # 2. Volatilität (annualisiert)
        vol = daily_returns.std() * np.sqrt(252)
        
        # 3. Sharpe Ratio (Annahme: Risk-Free Rate = 0, da Cash bereits in der Strategie (im Portfolio) steckt)
        sharpe = (daily_returns.mean() / daily_returns.std()) * np.sqrt(252) if daily_returns.std() != 0 else 0
        
        # 4. Maximum Drawdown
        peak = equity_curve.expanding(min_periods=1).max()
        drawdown = (equity_curve / peak) - 1
        mdd = drawdown.min()
        
        # 5. Sortino Ratio (Fokus auf Downside-Risiko)
        downside_returns = daily_returns[daily_returns < 0]
        downside_std = downside_returns.std() * np.sqrt(252)
        sortino = (daily_returns.mean() * 252) / downside_std if downside_std != 0 else np.nan
        
        # 6. Calmar Ratio (Verhältnis Rendite zu Max Drawdown)
        calmar = cagr / abs(mdd) if mdd != 0 else np.nan
        
        # 7. Anzahl der Trades (Regime-Wechsel)
        # Wir messen, wie oft das Modell von 0 auf 1 oder umgekehrt springt
        if col in trades_df.columns:
            # Signaländerungen zählen (Absolutwert der Differenz)
            switches = trades_df[col].diff().abs().sum()
        else:
            switches = 0 # Buy & Hold hat 0 Wechsel
        
        # 8. Gesamte Transaktionskosten am Ende des Zeitraums extrahieren
        if col in costs_df.columns:
            total_fees = costs_df[col].iloc[-1]
        else:
            total_fees = 0.0
            
        stats.append({
            'Strategie': col.replace('_', ' '),
            'Total Return': f"{total_return:.2%}",
            'CAGR (p.a.)': f"{cagr:.2%}",
            'Volatilität': f"{vol:.2%}",
            'Max Drawdown': f"{mdd:.2%}",
            'Sharpe Ratio': round(sharpe, 2),
            'Sortino Ratio': round(sortino, 2),
            'Calmar Ratio': round(calmar, 2),
            'Regime-Wechsel': int(switches),
            'Gesamtkosten (Gebühren)': f"{total_fees:.2%}"
        })
    
    return pd.DataFrame(stats).set_index('Strategie')

# --- 3. DYNAMISCHE ZUORDNUNG DER SIGNALE ---

# Wir erstellen automatisch ein DataFrame, das die Signale so benennt wie die Strategien
signals_to_count = pd.DataFrame(index=test_df.index)

# Wir suchen im test_df nach allen Spalten, die auf _Signal enden
signal_columns = [c for c in test_df.columns if c.endswith('_Signal')]

for sig_col in signal_columns:
    # Extrahiere Modellnamen (z.B. 'LSTM' aus 'LSTM_Signal')
    model_name = sig_col.rsplit('_', 1)[0]
    # Mappe das Signal auf den Modellnamen für die evaluate-Funktion
    signals_to_count[model_name] = test_df[sig_col]

# 4. Evaluation ausführen
evaluation_table = evaluate_strategies(backtesting_results, signals_to_count, backtesting_transaction_costs)

# 5. Anzeige und Persistierung
print("\n--- UMFASSENDE EVALUATION (DYNAMISCH) ---")
display(evaluation_table)

# Tabelle in eine Markdown-Dateie schreiben
evaluation_table.to_markdown('../assets/evaluation_table.md', index=True)
print("\nEvaluationstabelle erfolgreich unter ../assets/evaluation_table.md persistiert.")

#--- Wir erhalten in diesem Schritt neben df und test_df sowie backtesting_results trades_df mit binären Handelssignalen ---


--- UMFASSENDE EVALUATION (DYNAMISCH) ---


,Total Return,CAGR (p.a.),Volatilität,Max Drawdown,Sharpe Ratio,Sortino Ratio,Calmar Ratio,Regime-Wechsel,Gesamtkosten (Gebühren)
Strategie,,,,,,,,,
Buy Hold,92.90%,9.80%,12.62%,-27.10%,0.81,1.04,0.36,0,0.00%
HMM,73.59%,8.17%,4.96%,-6.79%,1.61,1.49,1.20,29,3.00%
MS Univariate,154.84%,14.24%,6.33%,-6.25%,2.14,2.78,2.28,42,4.20%
MS Exo,152.51%,14.09%,6.43%,-5.44%,2.09,2.70,2.59,38,3.80%
LSTM,72.65%,8.08%,7.70%,-11.58%,1.05,1.26,0.70,70,7.00%
LSTM Unsupervised,69.11%,7.76%,8.34%,-15.95%,0.94,0.95,0.49,19,2.00%



Evaluationstabelle erfolgreich unter ../assets/evaluation_table.md persistiert.
